###### Simon Steffens, ETH Zurich, June 13. 2023
### Register physical points


This notebook is the first in 3 jupyter notebooks to create an .obj file that 
warps a texture such that it is correctly displayed in a dome VR setup.

In this notebook specifically, an empty canvas and a calibration image are 
loaded into the napari image viewer. Then, a collection of empty Points layers
are added to the viewer. For the *_proj Points layers, one iteratively adds 
points (clicks) on the empty canvas where a physical point is marked in the 
sphere (see pictures). The joints in the calibration image are also registered 
and saved in the *_gt points layers. Finally, these two collection of XY 
coordinates are saved as `mapping.npy`

In [1]:
import napari
import numpy as np
from skimage.io import imread

In [2]:
viewer = napari.Viewer()

In [3]:
# add empty canvas for registering points
viewer.add_image(np.full((1080,1920), .5 ))

<Image layer 'Image' at 0x7fa6b38a7400>

In [4]:
# add callibration image to resgister ground truth
cimg = imread('./circ_callibr_v8.png')
viewer.add_image(cimg, name='gt_callibr', opacity=.2)

<Image layer 'gt_callibr' at 0x7fa69f3bafa0>

In [5]:
# add empty points layers for registering points, one for each ring (center point is ring_0)
n_rings = 15
for i in range(n_rings):
    col = [np.random.rand() for _ in range(3)]
    viewer.add_points(name=f'ring_{i}_proj', face_color=col, size=5)
    viewer.add_points(name=f'ring_{i}_gt', face_color=col, size=5)

/home/loaloa/anaconda3/envs/napari-env/lib/python3.9/site-packages/napari/layers/base/base.py:1632: RuntimeWarning: invalid value encountered in cast
  corners[:, displayed_axes] = data_bbox_clipped


1. Register physical landmarks in sphere and save in `ring_i_proj` layers. Click anywhere for non-visible points.
2. do the same for the callibration image and register `in ring_i_gt` layers
3. Take a screenshot of the napari window. Helpful for final alignment in Unity.

Note: Do not pan in the napari window for consistent registration. 

------

![alt physical_points_registration_1](physical_points_registration_1.jpg)
![alt physical_points_registration_1](physical_points_registration_2.jpg)

-------------
Run below to register a new mapping.

In [8]:
# iterate rings from inner to outer to construct numpy array of matching points
def map_points(n_rings):
    mapping = []
    # aggregate and validate registered points ring wise
    for i in range(n_rings):
        key_proj = f'ring_{i}_proj'
        key_gt = f'ring_{i}_gt'

        proj_data = viewer.layers[key_proj].data
        gt_data = viewer.layers[key_gt].data

        # get the points array from the ring layers and compare n points
        print(f"{key_proj}: n points: {proj_data.shape[0]}")
        print(f"{key_gt}: n points: {gt_data.shape[0]}")
        if proj_data.shape[0] != gt_data.shape[0]:
            print("n points differ, can't map !")
        else:
            mapping_i = np.stack((proj_data, gt_data), axis=1)
            mapping.append(mapping_i)
        print("----")
      
    # points-replicates-dim(n_rings*n_radians) - proj_vs_gt-dim(2) - YX-dim(2)
    print(f"mapping_flat.shape (points-replicates-dim - proj_vs_gt-dim - YX-dim: {np.concatenate(mapping, axis=0).shape}")
    return mapping

In [9]:
mapping = map_points(n_rings)

ring_0_proj: n points: 0
ring_0_gt: n points: 0
----
ring_1_proj: n points: 0
ring_1_gt: n points: 0
----
ring_2_proj: n points: 0
ring_2_gt: n points: 0
----
ring_3_proj: n points: 0
ring_3_gt: n points: 0
----
ring_4_proj: n points: 0
ring_4_gt: n points: 0
----
ring_5_proj: n points: 0
ring_5_gt: n points: 0
----
ring_6_proj: n points: 0
ring_6_gt: n points: 0
----
ring_7_proj: n points: 0
ring_7_gt: n points: 0
----
ring_8_proj: n points: 0
ring_8_gt: n points: 0
----
ring_9_proj: n points: 0
ring_9_gt: n points: 0
----
ring_10_proj: n points: 0
ring_10_gt: n points: 0
----
ring_11_proj: n points: 0
ring_11_gt: n points: 0
----
ring_12_proj: n points: 0
ring_12_gt: n points: 0
----
ring_13_proj: n points: 0
ring_13_gt: n points: 0
----
ring_14_proj: n points: 0
ring_14_gt: n points: 0
----
mapping_flat.shape (points-replicates-dim - proj_vs_gt-dim - YX-dim: (0, 2, 2)


In [10]:
# save registered points
# points-replicates-dim(n_rings*n_radians) - proj_vs_gt-dim(2) - YX-dim(2)
np.save('mapping_v6.npy', mapping_flat)

In [15]:
# check
mapping = np.load('mapping_v6.npy')
mapping.shape, mapping

((411, 2, 2),
 array([[[ 854.1814656 ,  961.71609815],
         [ 538.85853138,  959.91825687]],
 
        [[ 828.69670032,  959.5000316 ],
         [ 499.49408924,  959.91825687]],
 
        [[ 829.80473359,  946.20363232],
         [ 500.45086387,  967.84581813]],
 
        ...,
 
        [[ 536.1759162 , 1133.46125549],
         [  90.41449519,  659.64999409]],
 
        [[ 527.31165001, 1074.73549201],
         [  41.08159588,  753.25596979]],
 
        [[ 520.66345037,  997.17316289],
         [  10.02013996,  854.31110673]]]))